In [1]:
import os

from dotenv import load_dotenv

load_dotenv()  # Returns a boolean indicating whether the .env file was found and loaded successfully
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

# Add project root to sys.path so `from src...` imports work,
# regardless of where Jupyter's working directory happens to be
project_root = Path.cwd().parent  # assumes notebook runs from notebooks/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [3]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# invoked llm to test if the API key is working
llm.invoke("What is the capital of France?")

AIMessage(content='The capital of France is Paris.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 14, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c6b5b9a933', 'id': 'chatcmpl-E8xCMZREMPtSnWSDQMdgHFEizXE0y', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fca25-966b-75c1-a96e-a1cee9bd38f5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 7, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
from src.rag.retrieve import retrieve_relevant_documents
from src.data.state import AgriChainState

# Build a minimal test state — just enough for this function to read/write
test_state = {
    "complaint_text": "the tomatoes arrived bruised and moldy",
}

result = retrieve_relevant_documents(test_state)  # type: ignore

# Inspect what came back
for doc in result["retrieved_documents"]: # type: ignore
    print(doc["score"], doc["doc_type"], doc["content"][:80])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

1.1126602 supplier_info Supplier SUP-5806 - Asia Region: Specializes in tomatoes. Quality score: 88/100.
1.1330247 supplier_info Supplier SUP-5814 - Europe Region: Specializes in tomatoes. Quality score: 76/10
1.1471378 supplier_info Supplier SUP-12905 - Europe Region: Specializes in tomatoes. Quality score: 66/1
1.3364317 supplier_info Supplier SUP-6479 Profile: Certified organic producer in Europe. Capacity: 218 t
1.4174404 supplier_info Supplier SUP-6050 Profile: Certified organic producer in North America. Capacity


In [7]:
import json
from collections import Counter

with open("../data/raw/knowledge_base.json") as f:
    kb = json.load(f)

print(Counter(doc["doc_type"] for doc in kb))

Counter({'supplier_info': 200, 'sop': 50, 'resolution_guide': 24})


In [8]:
from src.models.classify import predict_category

# create a test complaint text
test_complaint_text = "the invoice charged me twice for the same order"

category = predict_category(test_complaint_text)  # type: ignore
print(f"Predicted category: {category}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
Predicted category: Pricing Error


In [9]:
# test analyzer
from src.agents.analyzer import analyze_severity

# Minimal state — only what this function actually reads
test_state = {
    "complaint_text": "the tomatoes arrived bruised and moldy",
}
test_state_two = {
    "complaint_text": "metal shavings, someone could get hurt",
}
result = analyze_severity(test_state)
result_two = analyze_severity(test_state_two)
print("Severity:", result["severity"])
print("Reasoning:", result["severity_reasoning"])
print("Severity:", result_two["severity"])
print("Reasoning:", result_two["severity_reasoning"])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Severity: high
Reasoning: The complaint involves perishable goods (tomatoes) that are bruised and moldy, indicating they may spoil quickly, which poses a risk of further loss if not addressed promptly. The financial impact could be significant if the customer is unable to use the product, and there are potential safety concerns related to consuming moldy food. Additionally, the customer's satisfaction and trust in the company may be jeopardized, affecting the long-term relationship.
Severity: high
Reasoning: The presence of metal shavings poses a significant safety risk, as it could lead to injury if the goods are used or consumed. This raises immediate concerns for customer safety and potential liability for the company. While the financial impact may not be directly quantifiable, the risk of harm to customers and the potential for damage to the company's reputation and customer relationships is substantial.


In [10]:
# test investogator
from src.agents.investigator import investigate

# Minimal state — only what this function actually reads
test_state = {
    "complaint_text": "the tomatoes arrived bruised and moldy",
}
result = investigate(test_state)
print("Retrieved documents:")
for doc in result["retrieved_documents"]:  # type: ignore
    print(doc["score"], doc["doc_type"], doc["content"][:80])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Retrieved documents:
1.1126602 supplier_info Supplier SUP-5806 - Asia Region: Specializes in tomatoes. Quality score: 88/100.
1.1330247 supplier_info Supplier SUP-5814 - Europe Region: Specializes in tomatoes. Quality score: 76/10
1.1471378 supplier_info Supplier SUP-12905 - Europe Region: Specializes in tomatoes. Quality score: 66/1
1.3364317 supplier_info Supplier SUP-6479 Profile: Certified organic producer in Europe. Capacity: 218 t
1.4174404 supplier_info Supplier SUP-6050 Profile: Certified organic producer in North America. Capacity


In [11]:
# test planner
from src.agents.planner import plan_resolution
from src.agents.analyzer import analyze_severity
from src.agents.investigator import investigate

test_state = {
    "complaint_text": "the tomatoes arrived bruised and moldy",
}
test_state = analyze_severity(test_state)
test_state = investigate(test_state)

result = plan_resolution(test_state)
print("Resolution plan:")
print(result["resolution_plan"])  # type: ignore

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Resolution plan:
{'plan_summary': "To address the complaint regarding bruised and moldy tomatoes, we will initiate a refund for the affected order, investigate the supplier's quality control processes, and implement stricter quality checks for future shipments. Additionally, we will communicate with the customer to ensure their satisfaction and prevent similar issues in the future.", 'steps': ['Initiate a full refund for the customer for the affected order of tomatoes.', 'Contact the supplier (SUP-5806) to investigate the quality control processes and delivery conditions that led to the bruising and mold.', 'Review the quality score and known issues of the supplier to assess if they are still a viable option for future orders.', 'Consider sourcing tomatoes from alternative suppliers with higher quality scores, such as SUP-6050, who has a reliability rating of 4.6/5 and specializes in tomatoes.', 'Implement a new quality assurance protocol that includes inspecting all incoming shipments

In [ ]:
# Test drafting a customer response
from src.agents.communicator import draft_customer_response
from src.agents.planner import plan_resolution
from src.agents.analyzer import analyze_severity
from src.agents.investigator import investigate

test_state = {
    "complaint_text": "the tomatoes arrived bruised and moldy",}
test_state = analyze_severity(test_state)
test_state = investigate(test_state)
test_state = plan_resolution(test_state)
result = draft_customer_response(test_state)

print("Customer response:")
print(result["customer_response"])  # type: ignore
print("error_messgae:", result.get("error_message", "No error message"))  # type: ignore

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Customer response:
Subject: Our Apologies and Resolution for Your Recent Order

Dear [Customer's Name],

Thank you for reaching out to us regarding the condition of the tomatoes you received. We sincerely apologize for the inconvenience caused by the bruised and moldy produce. Your satisfaction is very important to us, and we take your concerns seriously.

To address this issue, we have initiated a full refund for your order. You should see this reflected in your account shortly. 

Additionally, we are taking the following steps to ensure that this does not happen again:
1. We are contacting our supplier to investigate their quality control processes and the delivery conditions that may have contributed to the issue.
2. We will review the supplier's quality score and any known issues to determine if they remain a viable option for future orders.
3. Stricter quality checks will be implemented before shipment, including thorough visual inspections and temperature control measures during 

In [ ]:
# Test escalation manager
from src.agents.communicator import draft_customer_response
from src.agents.planner import plan_resolution
from src.agents.analyzer import analyze_severity
from src.agents.investigator import investigate
from src.agents.escalation_manager import identify_escalation

test_state = {
    "complaint_text": "the rotten tomatoes arrived in my order and I am now dead",
}
test_state = analyze_severity(test_state)
test_state = investigate(test_state)
test_state = plan_resolution(test_state)
test_state = draft_customer_response(test_state)
result = identify_escalation(test_state)
print("Escalation needed:", result["escalate"])  # type: ignore
print("Escalation reason:", result.get("escalation_reason", "No escalation reason"))  # type: ignore

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Escalation needed: True
Escalation reason: Critical severity complaint


In [13]:
from src.graph import app
import json

test_state = {
    "complaint_text": "the rotten tomatoes arrived in my order and I am now dead",
}

result = app.invoke(test_state)

print(json.dumps(result, indent=2, default=str))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{
  "complaint_text": "the rotten tomatoes arrived in my order and I am now dead",
  "severity": "critical",
  "severity_reasoning": "The complaint involves rotten tomatoes, which are perishable goods that can spoil quickly, posing a significant risk to food safety. The phrase 'I am now dead' suggests a serious health concern, potentially indicating food poisoning or severe allergic reaction, which elevates the severity to critical. Additionally, this situation could lead to legal implications and a severe impact on customer trust and relationship.",
  "retrieved_documents": [
    {
      "content": "Supplier SUP-5806 - Asia Region: Specializes in tomatoes. Quality score: 88/100. Average delivery time: 2 days. Known issues: excellent track record. Preferred for large volume orders.",
      "score": "1.0009711",
      "doc_id": "KB-00185",
      "doc_type": "supplier_info",
      "category": "Supplier",
      "last_updated": "2025-08-06"
    },
    {
      "content": "Supplier SUP-12905

In [15]:
# Test the entire graph on a few real complaints
import pandas as pd
import json
from src.graph import app

# 1. Load the real test complaints
test_df = pd.read_csv("../data/raw/complaints_test.csv")

# 2. Randomly sample 5 rows — use a fixed random_state so it's reproducible,
#    not a different 5 complaints every time you rerun the notebook
sample_complaints = test_df.sample(n=5, random_state=42)

# 3. Loop through them, running each through the graph
reports = []
for index, row in sample_complaints.iterrows():
    result = app.invoke({"complaint_text": row["complaint_text"]})
    complaint_id = row["complaint_id"]
    actual_category = row["category"]
    actual_severity = row['priority']

    # what goes into each report? your turn to decide the shape
    report_dict = {
        "complaint_id": complaint_id,
        "complaint_text": result.get("complaint_text"),
        "severity": result.get("severity"),
        "actual_severity": actual_severity,
        "predicted_category": result.get("predicted_category"),
        "actual_category": actual_category,
        "resolution_plan": result.get("resolution_plan"),
        "customer_response": result.get("customer_response"),
        "escalation_needed": result.get("escalate"),
        "escalation_reason": result.get("escalation_reason", "No escalation reason"),
    }
    reports.append(report_dict)

print("Reports:")
for report in reports:
    print(json.dumps(report, indent=2, default=str))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Reports:
{
  "complaint_id": "CMP-000014",
  "complaint_text": "Order ORD-891930 contains cucumbers that are already showing signs of spoilage upon arrival.",
  "severity": "high",
  "actual_severity": "critical",
  "predicted_category": null,
  "actual_category": "Quality Issues",
  "resolution_plan": {
    "plan_summary": "To address the customer's complaint regarding spoilage of cucumbers in order ORD-891930, we will initiate a refund process, investigate the supplier's quality control measures, and implement stricter quality checks for future orders. Additionally, we will consider switching to a higher-rated supplier to ensure better product quality.",
    "steps": [
      "Apologize to the customer for the inconvenience caused by the spoiled cucumbers and assure them that their complaint is being taken seriously.",
      "Initiate a refund process for the affected order (ORD-891930) to compensate the customer for the spoiled product.",
      "Contact Supplier SUP-6668 to inquire a

In [ ]:
import json

with open("../data/processed/sample_resolution_reports.json", "w") as f:
    json.dump(reports, f, indent=2, default=str)

print(f"Saved {len(reports)} reports.")